# Supply Chain Optimization: Model Notebook
**Mohammed Islam Hadjoudj** · First Edition (2026) · companion package v1.1

Every companion model in chapter order. Run the first cell once to install the libraries.

Tested with Python 3.12.3, PuLP 4.0.0, HiGHS 1.13.1, OR-Tools 9.15, pandas 3.0.2, NumPy 2.4.4, SciPy 1.17.1.

In [ ]:
%pip install "pulp[highs]" ortools scipy numpy pandas openpyxl -q

## Chapter 1: Two-product plant
Maximize contribution under two department-hour limits. Book answer: 20 Standard, 15 Premium, $1,700.

In [ ]:
from pulp import LpProblem, LpMaximize, LpSolveStatus, HiGHS, value

model = LpProblem("two_product_plant", LpMaximize)
x = model.add_variable("standard", lowBound=0)
y = model.add_variable("premium", lowBound=0)
model += 40 * x + 60 * y, "contribution"
model += 2 * x + 4 * y <= 100, "machining_hours"
model += 3 * x + 2 * y <= 90, "finishing_hours"

result = model.solve(HiGHS(msg=False))
if result.status == LpSolveStatus.Optimal:
    print("Standard:", value(x))
    print("Premium :", value(y))
    print("Profit  :", value(model.objective))


## Chapter 2: Five-day work week staffing
Minimize pickers when each works 5 consecutive days. Book answer: 16 pickers (LP relaxation 15.33).

In [ ]:
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
required = [12, 10, 11, 14, 13, 9, 7]

def solve(integer=True):
    m = LpProblem("staffing", LpMinimize)
    cat = "Integer" if integer else "Continuous"
    start = {d: m.add_variable(f"start_{days[d]}", lowBound=0, cat=cat) for d in range(7)}
    m += lpSum(start.values())
    for t in range(7):               # day t is covered by starts on t, t-1, ..., t-4
        m += lpSum(start[(t - k) % 7] for k in range(5)) >= required[t], f"cover_{days[t]}"
    m.solve(HiGHS(msg=False))
    return value(m.objective), {days[d]: value(start[d]) for d in range(7)}

lp_total, _ = solve(integer=False)
ip_total, plan = solve(integer=True)
print(f"LP relaxation: {lp_total:.2f} pickers")
print(f"Integer optimum: {ip_total:.0f} pickers")
print("Starts by day:", {d: int(round(v)) for d, v in plan.items()})


## Chapter 3: Shadow prices and the dual
Read shadow prices and slacks, then solve the dual. Book answers: $12.50 and $5.00, dual value $1,700.

In [ ]:
from pulp import LpProblem, LpMaximize, LpMinimize, HiGHS, value

m = LpProblem("plant", LpMaximize)
x = m.add_variable("standard", lowBound=0)
y = m.add_variable("premium", lowBound=0)
m += 40 * x + 60 * y
m += 2 * x + 4 * y <= 100, "machining_hours"
m += 3 * x + 2 * y <= 90, "finishing_hours"
m.solve(HiGHS(msg=False))
for c in m.constraints():
    # HiGHS reports duals of this maximization as negative. Other solvers may
    # use the opposite sign. Interpret with the objective sense and constraint type.
    print(f"{c.name}: dual {c.pi:+.2f}, slack {round(c.slack + 0.0, 2)}")

# Verify the interpretation by re-solving with one more machining hour
m2 = LpProblem("plant_plus_one", LpMaximize)
x2 = m2.add_variable("standard", lowBound=0); y2 = m2.add_variable("premium", lowBound=0)
m2 += 40 * x2 + 60 * y2
m2 += 2 * x2 + 4 * y2 <= 101
m2 += 3 * x2 + 2 * y2 <= 90
m2.solve(HiGHS(msg=False))
print(f"With 101 machining hours: {value(m2.objective):.2f} (up {value(m2.objective) - 1700:.2f})")

d = LpProblem("dual", LpMinimize)
u = d.add_variable("machining_price", lowBound=0)
v = d.add_variable("finishing_price", lowBound=0)
d += 100 * u + 90 * v
d += 2 * u + 3 * v >= 40
d += 4 * u + 2 * v >= 60
d.solve(HiGHS(msg=False))
print(f"Dual: u = {value(u):.2f}, v = {value(v):.2f}, value = {value(d.objective):.0f}")


## Chapter 4: Cost against carbon: the epsilon-constraint method
Truck versus rail for 1,000 tons under a series of CO2 caps. Book answers: $40,000 uncapped, $44,286 at 80 t, $50,500 at the 51 t minimum.

In [ ]:
from pulp import LpProblem, LpMinimize, HiGHS, value

for cap in [None, 80, 70, 60, 51]:
    m = LpProblem("modes", LpMinimize)
    truck = m.add_variable("truck_tons", lowBound=0)
    rail = m.add_variable("rail_tons", lowBound=0, upBound=700)
    m += 40 * truck + 55 * rail
    m += truck + rail == 1000
    if cap is not None:
        m += 0.10 * truck + 0.03 * rail <= cap
    m.solve(HiGHS(msg=False))
    co2 = 0.10 * truck.value() + 0.03 * rail.value()
    print(f"cap {cap}: cost ${value(m.objective):,.0f}, rail {rail.value():.0f} t, CO2 {co2:.1f} t")


## Chapter 4: Product launch with setups, minimum lots, and a dependency
Binary run decisions, linking constraints, and "C requires A". Book answers: $4,880 without the rule, $4,680 with it.

In [ ]:
from pulp import LpProblem, LpMaximize, HiGHS, lpSum, value

products = ["A", "B", "C"]
margin = {"A": 18, "B": 36, "C": 60}
hours  = {"A": 2, "B": 3, "C": 4}
setup  = {"A": 800, "B": 1000, "C": 2400}
demand = {"A": 150, "B": 120, "C": 90}
min_lot, capacity = 50, 600

def solve(c_requires_a):
    m = LpProblem("launch", LpMaximize)
    make = {p: m.add_variable(f"make_{p}", lowBound=0, cat="Integer") for p in products}
    run  = {p: m.add_variable(f"run_{p}", cat="Binary") for p in products}
    m += lpSum(margin[p] * make[p] - setup[p] * run[p] for p in products)
    m += lpSum(hours[p] * make[p] for p in products) <= capacity
    for p in products:
        m += make[p] <= demand[p] * run[p]     # linking (gate)
        m += make[p] >= min_lot * run[p]       # minimum lot
    if c_requires_a:
        m += run["C"] <= run["A"]
    m.solve(HiGHS(msg=False))
    return value(m.objective), {p: int(make[p].value()) for p in products}

print("Without the rule:", solve(False))
print("C requires A    :", solve(True))


## Chapter 5: Transportation problem with an optional lane cap
Two plants, three depots. Book answers: $5,650, and capping lane P2-D3 at 200 raises cost to $6,000 (shadow price $7/unit).

In [ ]:
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value

supply = {"P1": 500, "P2": 400}
demand = {"D1": 300, "D2": 350, "D3": 250}
cost = {("P1","D1"): 8, ("P1","D2"): 6, ("P1","D3"): 10,
        ("P2","D1"): 9, ("P2","D2"): 7, ("P2","D3"): 4}

def solve(lane_caps=None):
    m = LpProblem("distribution", LpMinimize)
    x = {k: m.add_variable(f"x_{k[0]}_{k[1]}", lowBound=0) for k in cost}
    m += lpSum(cost[k] * x[k] for k in cost)
    for i in supply:
        m += lpSum(x[i, j] for j in demand) <= supply[i]
    for j in demand:
        m += lpSum(x[i, j] for i in supply) >= demand[j]
    for k, cap in (lane_caps or {}).items():
        m += x[k] <= cap, f"cap_{k[0]}_{k[1]}"
    m.solve(HiGHS(msg=False))
    caps = {c.name: c.pi for c in m.constraints() if c.name.startswith("cap_")}   # signed dual
    return value(m.objective), {k: x[k].value() for k in cost if x[k].value() > 0}, caps

print("Base plan:", solve())
print("Lane P2-D3 capped at 200:", solve({("P2", "D3"): 200}))


## Chapter 6: Shortest path, assignment, and minimum-cost flow
Book answers: shortest path 10 (Exercise 6.1 network), assignment 9 driver-hours, minimum-cost flow $1,020.

In [ ]:
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value
H = lambda: HiGHS(msg=False)

# Shortest path as a unit flow from S to T
arcs = {("S","A"):3, ("S","B"):5, ("A","B"):1, ("A","C"):6, ("B","C"):4,
        ("B","D"):3, ("D","C"):1, ("C","T"):2, ("D","T"):6}
nodes = {n for a in arcs for n in a}
m = LpProblem("shortest_path", LpMinimize)
f = {a: m.add_variable(f"f_{a[0]}_{a[1]}", lowBound=0) for a in arcs}
m += lpSum(arcs[a] * f[a] for a in arcs)
for n in nodes:
    b = 1 if n == "S" else (-1 if n == "T" else 0)
    m += lpSum(f[a] for a in arcs if a[0] == n) - lpSum(f[a] for a in arcs if a[1] == n) == b
m.solve(H()); print("Shortest path cost:", value(m.objective))

# Assignment: trucks to routes
cost = {("A","R1"):9, ("A","R2"):2, ("A","R3"):7, ("B","R1"):6, ("B","R2"):4,
        ("B","R3"):3, ("C","R1"):5, ("C","R2"):8, ("C","R3"):1}
m = LpProblem("assignment", LpMinimize)
x = {k: m.add_variable(f"x_{k[0]}_{k[1]}", cat="Binary") for k in cost}
m += lpSum(cost[k] * x[k] for k in cost)
for t in "ABC":
    m += lpSum(x[k] for k in cost if k[0] == t) == 1
for r in ("R1", "R2", "R3"):
    m += lpSum(x[k] for k in cost if k[1] == r) == 1
m.solve(H()); print("Assignment:", [k for k in cost if x[k].value() > 0.5], value(m.objective))

# Minimum-cost flow: two plants, a hub, three customers
supply = {"A": 120, "B": 80, "H": 0, "C1": -60, "C2": -70, "C3": -70}
arc = {("A","H"):(2,100), ("B","H"):(3,100), ("A","C1"):(6,50), ("B","C3"):(5,40),
       ("H","C1"):(3,80), ("H","C2"):(2,80), ("H","C3"):(4,80)}
m = LpProblem("min_cost_flow", LpMinimize)
f = {k: m.add_variable(f"f_{k[0]}_{k[1]}", lowBound=0, upBound=arc[k][1]) for k in arc}
m += lpSum(arc[k][0] * f[k] for k in arc)
for n, b in supply.items():
    m += lpSum(f[k] for k in arc if k[0] == n) - lpSum(f[k] for k in arc if k[1] == n) == b
m.solve(H())
print("Min-cost flow:", value(m.objective), {k: f[k].value() for k in arc if f[k].value() > 0})


## Chapter 7: Nearest neighbor, 2-opt, and Clarke-Wright savings
Book answers: nearest neighbor 64 km, after 2-opt 57 km, savings plan 76 km with three trucks.

In [ ]:
D = [[0, 8, 9,10, 9,10,10],     # row/column 0 is the depot DC, then customers A-F
     [8, 0, 8,14, 7,10,18],
     [9, 8, 0, 8, 1,17,16],
     [10,14,8, 0, 9,21,11],
     [9, 7, 1, 9, 0,16,16],
     [10,10,17,21,16,0,19],
     [10,18,16,11,16,19,0]]
NAMES = ["DC", "A", "B", "C", "D", "E", "F"]
length = lambda r: sum(D[r[i]][r[i + 1]] for i in range(len(r) - 1))
show = lambda r: "->".join(NAMES[i] for i in r)

def nearest_neighbor():
    route, left = [0], set(range(1, len(D)))
    while left:
        nxt = min(left, key=lambda j: (D[route[-1]][j], j))
        route.append(nxt); left.remove(nxt)
    return route + [0]

def two_opt(route):
    improved = True
    while improved:
        improved = False
        for i in range(1, len(route) - 2):
            for k in range(i + 1, len(route) - 1):
                new = route[:i] + route[i:k + 1][::-1] + route[k + 1:]
                if length(new) < length(route):
                    route, improved = new, True
    return route

def clarke_wright(demand, capacity):
    routes = {c: [c] for c in range(1, len(D))}
    savings = sorted(((D[0][i] + D[0][j] - D[i][j], i, j)
                      for i in range(1, len(D)) for j in range(i + 1, len(D))), reverse=True)
    find = lambda c: next(k for k, r in routes.items() if c in r)
    for s, i, j in savings:
        ki, kj = find(i), find(j)
        if ki == kj: continue
        ri, rj = routes[ki], routes[kj]
        if sum(demand[c] for c in ri + rj) > capacity: continue
        if ri[-1] == i and rj[0] == j: new = ri + rj
        elif ri[0] == i and rj[-1] == j: new = rj + ri
        elif ri[-1] == i and rj[-1] == j: new = ri + rj[::-1]
        elif ri[0] == i and rj[0] == j: new = ri[::-1] + rj
        else: continue
        routes[ki] = new; del routes[kj]
    return [[0] + r + [0] for r in routes.values()]

nn = nearest_neighbor();  print("Nearest neighbor:", show(nn), length(nn), "km")
opt = two_opt(nn);        print("After 2-opt     :", show(opt), length(opt), "km")
plan = clarke_wright(demand=[0, 4, 3, 5, 2, 6, 4], capacity=10)
for r in plan: print("  truck:", show(r), length(r), "km")
print("Savings plan total:", sum(length(r) for r in plan), "km")


## Chapter 7: Capacitated vehicle routing with OR-Tools
Book answer: three trucks, 76 km in total.

In [ ]:
from ortools.constraint_solver import pywrapcp, routing_enums_pb2

D = [[0, 8, 9,10, 9,10,10], [8, 0, 8,14, 7,10,18], [9, 8, 0, 8, 1,17,16],
     [10,14,8, 0, 9,21,11], [9, 7, 1, 9, 0,16,16], [10,10,17,21,16,0,19],
     [10,18,16,11,16,19,0]]
demand, capacity, trucks = [0, 4, 3, 5, 2, 6, 4], 10, 3
names = ["DC", "A", "B", "C", "D", "E", "F"]

manager = pywrapcp.RoutingIndexManager(len(D), trucks, 0)
routing = pywrapcp.RoutingModel(manager)
def distance(i, j):
    return D[manager.IndexToNode(i)][manager.IndexToNode(j)]
routing.SetArcCostEvaluatorOfAllVehicles(routing.RegisterTransitCallback(distance))
def load(i):
    return demand[manager.IndexToNode(i)]
routing.AddDimensionWithVehicleCapacity(routing.RegisterUnaryTransitCallback(load),
                                        0, [capacity] * trucks, True, "Load")
params = pywrapcp.DefaultRoutingSearchParameters()
params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.SAVINGS
params.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
params.time_limit.seconds = 5
solution = routing.SolveWithParameters(params)

total = 0
for v in range(trucks):
    i, stops, km = routing.Start(v), [], 0
    while not routing.IsEnd(i):
        stops.append(names[manager.IndexToNode(i)])
        nxt = solution.Value(routing.NextVar(i))
        km += routing.GetArcCostForVehicle(i, nxt, v)
        i = nxt
    print(f"Truck {v + 1}: {' -> '.join(stops)} -> DC ({km} km)")
    total += km
print("Total distance:", total, "km")


## Chapter 8: Set covering and p-median location
Book answers: covering within 45 km needs T2 and T6, and the 2-median is T1 and T3 (22.9 km on average).

In [ ]:
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value

towns = ["T1", "T2", "T3", "T4", "T5", "T6"]
pop = [40, 25, 30, 15, 35, 20]
dist = [[0,30,55,80,60,95], [30,0,35,60,50,80], [55,35,0,30,45,50],
        [80,60,30,0,55,40], [60,50,45,55,0,45], [95,80,50,40,45,0]]
n = len(towns)

def covering(radius):
    m = LpProblem("covering", LpMinimize)
    y = {j: m.add_variable(f"open_{towns[j]}", cat="Binary") for j in range(n)}
    m += lpSum(y.values())
    for i in range(n):
        m += lpSum(y[j] for j in range(n) if dist[i][j] <= radius) >= 1
    m.solve(HiGHS(msg=False))
    return [towns[j] for j in range(n) if y[j].value() > 0.5]

def p_median(p):
    m = LpProblem("p_median", LpMinimize)
    y = {j: m.add_variable(f"open_{towns[j]}", cat="Binary") for j in range(n)}
    x = {(i, j): m.add_variable(f"serve_{i}_{j}", lowBound=0) for i in range(n) for j in range(n)}
    m += lpSum(pop[i] * dist[i][j] * x[i, j] for i in range(n) for j in range(n))
    m += lpSum(y.values()) == p
    for i in range(n):
        m += lpSum(x[i, j] for j in range(n)) == 1
        for j in range(n):
            m += x[i, j] - y[j] <= 0
    m.solve(HiGHS(msg=False))
    return [towns[j] for j in range(n) if y[j].value() > 0.5], value(m.objective) / sum(pop)

print("Covering within 45 km:", covering(45))
for p in (1, 2, 3):
    sites, avg = p_median(p)
    print(f"{p}-median: {sites}, average distance {avg:.1f} km")


## Chapter 9: Slotting, forward pick allocation, and dock door assignment
Book answers: slotting 16,390 -> 12,290 m/day, square-root allocation 17.1/8.6/4.3 m3 (1.63 replenishments/week), doors 610 -> 490 m.

In [ ]:
import math
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value

# Slotting by velocity
picks = {"A":120, "B":35, "C":260, "D":15, "E":80, "F":190, "G":45, "H":10}
slot_m = [10, 14, 18, 22, 26, 30, 34, 38]
walk = lambda order: sum(picks[p] * slot_m[i] for i, p in enumerate(order))
alpha = sorted(picks); velocity = sorted(picks, key=picks.get, reverse=True)
print("Alphabetical:", walk(alpha), "m/day | By velocity:", walk(velocity), "m/day", velocity)

# Square-root allocation of forward space
flow, space = [16, 4, 1], 30
roots = [math.sqrt(f) for f in flow]
alloc = [space * r / sum(roots) for r in roots]
print("Forward space:", [round(a, 1) for a in alloc],
      "replenishments/week:", round(sum(f / a for f, a in zip(flow, alloc)), 2))

# Dock door assignment
travel = [[100,120,300,320], [110,250,280,300], [300,280,120,260], [310,300,130,140]]
m = LpProblem("doors", LpMinimize)
x = {(t, d): m.add_variable(f"x_{t}_{d}", cat="Binary") for t in range(4) for d in range(4)}
m += lpSum(travel[t][d] * x[t, d] for t in range(4) for d in range(4))
for t in range(4): m += lpSum(x[t, d] for d in range(4)) == 1
for d in range(4): m += lpSum(x[t, d] for t in range(4)) == 1
m.solve(HiGHS(msg=False))
print("Door plan:", {f"trailer {t+1}": f"door {d+1}" for (t, d) in x if x[t, d].value() > 0.5},
      "total", value(m.objective), "m")


## Chapter 10: Moving average, exponential smoothing, and Holt's method
Book answers (scored on months 4-12, Holt initialized from months 1-3 only): MAD 8.9 (moving average), 13.1 (smoothing), 5.8 (Holt).

In [ ]:
demand = [120,132,126,140,135,150,144,158,152,165,160,172]
n = len(demand)

ma = [None] * n
for t in range(3, n):
    ma[t] = sum(demand[t-3:t]) / 3

alpha = 0.3
ses = [None] * n; ses[1] = demand[0]
for t in range(2, n):
    ses[t] = alpha * demand[t-1] + (1 - alpha) * ses[t-1]

a, b = 0.3, 0.2
level, trend = demand[0], (demand[2] - demand[0]) / 2   # uses months 1-3 only
holt = [None] * n
for t in range(1, n):
    holt[t] = level + trend
    new_level = a * demand[t] + (1 - a) * (level + trend)
    trend = b * (new_level - level) + (1 - b) * trend
    level = new_level

def score(f):
    e = [demand[t] - f[t] for t in range(3, n)]
    bias = sum(e) / len(e); mad = sum(abs(x) for x in e) / len(e)
    mape = 100 * sum(abs(demand[t] - f[t]) / demand[t] for t in range(3, n)) / len(e)
    return f"bias {bias:+.1f}, MAD {mad:.1f}, MAPE {mape:.1f}%"

for name, f in [("3-month MA", ma), ("Smoothing", ses), ("Holt", holt)]:
    print(f"{name:11s}: {score(f)}")
print("Holt forecast for month 13:", round(level + trend, 1))


## Chapter 11: EOQ, cost penalty, reorder point, and newsvendor ratio
Book answers: EOQ about 949 units, ordering 1,200 costs 2.8% more, reorder point 249.5.

In [ ]:
import math

def eoq(D, S, H): return math.sqrt(2 * D * S / H)
def total_cost(Q, D, S, H): return D / Q * S + H * Q / 2

D, S, H = 12000, 150, 4
q = eoq(D, S, H)
print(f"EOQ: {q:.0f} units, annual cost ${total_cost(q, D, S, H):,.0f}")
for Q in (600, 1200):
    pct = 100 * (total_cost(Q, D, S, H) / total_cost(q, D, S, H) - 1)
    print(f"Ordering {Q}: ${total_cost(Q, D, S, H):,.0f} ({pct:+.1f}%)")

d, sigma, L, z = 50, 15, 4, 1.65
print(f"Safety stock {z * sigma * math.sqrt(L):.1f}, reorder point {d * L + z * sigma * math.sqrt(L):.1f}")

def critical_ratio(price, cost, salvage):
    cu, co = price - cost, cost - salvage
    return cu / (cu + co)
print("Newsvendor critical ratio (Exercise 11.3 data):", critical_ratio(12, 5, 2))


## Chapter 12: Periodic review, fill rate, ABC, and space-constrained ordering
Book answers: order-up-to 632.1, fill rate 99.88%, ABC class A items P01, P04, P09, P06, and a space shadow price of about $55/m2.

In [ ]:
import math
from scipy.stats import norm

d, sigma, L, R, z = 50, 15, 4, 7, 1.65
print(f"Periodic review: S = {d*(R+L) + z*sigma*math.sqrt(R+L):.1f}")

G = lambda z: norm.pdf(z) - z * (1 - norm.cdf(z))       # standard normal loss function
sigma_L, Q = sigma * math.sqrt(L), 500
print(f"Fill rate at z=1.65, Q=500: {100*(1 - sigma_L*G(1.65)/Q):.2f}%")

items = {"P01":(1200,45), "P02":(300,12), "P03":(5000,2.5), "P04":(80,310), "P05":(2500,1.2),
         "P06":(650,30), "P07":(150,8), "P08":(9000,0.9), "P09":(400,55), "P10":(220,4.5)}
usage = {k: u * c for k, (u, c) in items.items()}; total = sum(usage.values()); cum = 0
for k in sorted(usage, key=usage.get, reverse=True):
    cum += usage[k]
    cls = "A" if cum / total <= 0.81 else ("B" if cum / total <= 0.95 else "C")
    print(f"  {k}: ${usage[k]:>9,.0f}  cumulative {100*cum/total:5.1f}%  class {cls}")

# Space-constrained order quantities: find the space price (lambda) by bisection
data = [(12000, 75, 3.0, 0.02), (6000, 120, 4.0, 0.05), (20000, 50, 1.5, 0.01)]   # D, S, H, m2/unit
W = 40
lo, hi = 0.0, 1000.0
for _ in range(100):
    lam = (lo + hi) / 2
    Q = [math.sqrt(2*D*S / (H + 2*lam*f)) for D, S, H, f in data]
    if sum(q * f for q, (D, S, H, f) in zip(Q, data)) > W: lo = lam
    else: hi = lam
print("Space price per m2:", round(lam, 2), "order quantities:", [round(q) for q in Q])


## Chapter 13: Quantity discounts and supplier selection
Book answers: order 2,000 at $9.30 ($48,610/year), and buy 5,000 each from suppliers S1 and S2 ($88,000).

In [ ]:
import math
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value

D, S, h = 5000, 100, 0.20
breaks = [(0, 10.00), (500, 9.60), (2000, 9.30)]
for i, (low, price) in enumerate(breaks):
    high = breaks[i + 1][0] - 1 if i + 1 < len(breaks) else float("inf")
    q = min(max(math.sqrt(2 * D * S / (h * price)), low), high)
    tc = D * price + D / q * S + h * price * q / 2
    print(f"${price:.2f}: best Q {q:.0f}, total ${tc:,.0f}")

sup = {"S1": (8.00, 6000, 4000, 0.030), "S2": (8.40, 5000, 2000, 0.010),
       "S3": (8.90, 8000, 1000, 0.005), "S4": (7.60, 4000, 6000, 0.050)}   # price, capacity, fixed, defects
m = LpProblem("suppliers", LpMinimize)
x = {k: m.add_variable(f"buy_{k}", lowBound=0) for k in sup}
y = {k: m.add_variable(f"use_{k}", cat="Binary") for k in sup}
m += lpSum(sup[k][0] * x[k] + sup[k][2] * y[k] for k in sup)
m += lpSum(x.values()) == 10000
for k in sup:
    m += x[k] <= sup[k][1] * y[k]
    m += x[k] >= 2000 * y[k]
m += lpSum(sup[k][3] * x[k] for k in sup) <= 0.02 * 10000
m.solve(HiGHS(msg=False))
print("Supplier plan:", {k: round(x[k].value()) for k in sup}, f"total ${value(m.objective):,.0f}")


## Chapter 14: Aggregate production plan and workforce planning
Book answers: workforce plan 26, 26, 26, 22 workers at $320,000.

In [ ]:
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value

# Aggregate plan (regular, overtime, inventory)
demand, cap, ot_cap = [400, 400, 800, 300], 500, 150
m = LpProblem("aggregate", LpMinimize)
T = range(4)
R = {t: m.add_variable(f"reg_{t}", lowBound=0, upBound=cap) for t in T}
O = {t: m.add_variable(f"ot_{t}", lowBound=0, upBound=ot_cap) for t in T}
I = {t: m.add_variable(f"inv_{t}", lowBound=0) for t in T}
m += lpSum(20 * R[t] + 30 * O[t] + 3 * I[t] for t in T)
for t in T:
    m += I[t] == (I[t - 1] if t else 0) + R[t] + O[t] - demand[t]
m.solve(HiGHS(msg=False))
print("Aggregate plan: regular", [R[t].value() for t in T], "overtime", [O[t].value() for t in T],
      f"cost ${value(m.objective):,.0f}")

# Workforce planning with hiring and layoffs
demand = [2000, 2600, 3200, 2200]
m = LpProblem("workforce", LpMinimize)
W = {t: m.add_variable(f"workers_{t}", lowBound=0, cat="Integer") for t in T}
Hh = {t: m.add_variable(f"hires_{t}", lowBound=0, cat="Integer") for t in T}
F = {t: m.add_variable(f"layoffs_{t}", lowBound=0, cat="Integer") for t in T}
I = {t: m.add_variable(f"inv_{t}", lowBound=0) for t in T}
m += lpSum(3000 * W[t] + 1500 * Hh[t] + 2000 * F[t] + 5 * I[t] for t in T)
for t in T:
    m += W[t] == (W[t - 1] if t else 22) + Hh[t] - F[t]
    m += I[t] == (I[t - 1] if t else 0) + 100 * W[t] - demand[t]
m.solve(HiGHS(msg=False))
print("Workforce plan:", [int(W[t].value()) for t in T], f"cost ${value(m.objective):,.0f}")


## Chapter 15: Wagner-Whitin, Silver-Meal, dispatching rules, and Johnson's rule
Book answers: Wagner-Whitin $730, Silver-Meal $770, Johnson sequence B, D, C, E, A with makespan 30.

In [ ]:
d, S, h = [60, 100, 10, 200, 50, 80], 200, 1
n = len(d)
F, choice = [0] + [None] * n, [None] * (n + 1)
for j in range(1, n + 1):                      # Wagner-Whitin dynamic program
    for i in range(1, j + 1):
        c = F[i - 1] + S + sum(h * d[t - 1] * (t - i) for t in range(i, j + 1))
        if F[j] is None or c < F[j]: F[j], choice[j] = c, i
plan, j = [], n
while j > 0: plan.append((choice[j], j)); j = choice[j] - 1
print("Wagner-Whitin:", F[n], "runs (first week, last week):", plan[::-1])

t, total = 0, 0                                # Silver-Meal heuristic
while t < n:
    best, end = None, t
    for e in range(t, n):
        avg = (S + sum(h * d[u] * (u - t) for u in range(t, e + 1))) / (e - t + 1)
        if best is None or avg <= best: best, end = avg, e
        else: break
    total += S + sum(h * d[u] * (u - t) for u in range(t, end + 1)); t = end + 1
print("Silver-Meal:", total)

jobs = {"J1": (6, 8), "J2": (2, 20), "J3": (8, 15), "J4": (3, 6), "J5": (4, 21)}
def evaluate(seq):
    t, flows, late = 0, [], []
    for j in seq:
        t += jobs[j][0]; flows.append(t); late.append(t - jobs[j][1])
    return sum(flows) / len(seq), max(late), sum(1 for x in late if x > 0)
for rule, key in [("SPT", lambda j: jobs[j][0]), ("EDD", lambda j: jobs[j][1])]:
    seq = sorted(jobs, key=key)
    print(rule, seq, "avg flow %.1f, max lateness %d, late jobs %d" % evaluate(seq))

two = {"A": (5, 2), "B": (1, 6), "C": (9, 7), "D": (3, 8), "E": (10, 4)}
front = sorted([j for j in two if two[j][0] < two[j][1]], key=lambda j: two[j][0])
back = sorted([j for j in two if two[j][0] >= two[j][1]], key=lambda j: two[j][1], reverse=True)
seq, t1, t2 = front + back, 0, 0
for j in seq:
    t1 += two[j][0]; t2 = max(t2, t1) + two[j][1]
print("Johnson:", seq, "makespan", t2)


## Chapter 16: Scenario planning, EVPI, and Monte Carlo simulation
Book answers: best quantity 1,000 (expected $50,400), EVPI $7,200, minimax regret 1,130 ($13,250). With lead-time variability, stockouts rise to about 18.6%.

In [ ]:
import numpy as np

cost, price, rush, rush_max, salvage = 40, 100, 85, 200, 15
scenarios = [(600, 0.3), (1000, 0.5), (1400, 0.2)]

def profit(q, d):
    rushed = min(max(0, d - q), rush_max)
    return price * min(q, d) + (price - rush) * rushed + salvage * max(0, q - d) - cost * q
expected = lambda q: sum(p * profit(q, d) for d, p in scenarios)

best = max(range(0, 1801, 10), key=expected)
foresight = sum(p * profit(d, d) for d, p in scenarios)
print(f"Average-based plan (960): expected ${expected(960):,.0f}")
print(f"Best quantity {best}: expected ${expected(best):,.0f}, EVPI ${foresight - expected(best):,.0f}")

best_case = {d: profit(d, d) for d, _ in scenarios}
max_regret = lambda q: max(best_case[d] - profit(q, d) for d, _ in scenarios)
q_mr = min(range(0, 2001), key=max_regret)          # every whole-number quantity
print(f"Minimax regret: produce {q_mr}, maximum regret ${max_regret(q_mr):,.0f}, expected ${expected(q_mr):,.0f}")

rng = np.random.default_rng(42); N = 200_000; reorder_point = 249.5
fixed = rng.normal(50, 15, (N, 4)).sum(axis=1)
lead = rng.integers(3, 6, N); days = rng.normal(50, 15, (N, 5))
variable = (days * (np.arange(5)[None, :] < lead[:, None])).sum(axis=1)
print(f"Stockout rate, fixed 4-day lead time : {(fixed > reorder_point).mean():.2%}")
print(f"Stockout rate, 3-5 day lead time     : {(variable > reorder_point).mean():.2%}")
print(f"Reorder point for 5% with variability: {np.percentile(variable, 95):.0f}")


## Chapter 17: Carbon abatement curve from an emissions-cap sweep
Book answers: $61,000 and 48.8 t, $61,500 at 40.2 t ($58/t), and $65,000 at 32.0 t ($426/t).

In [ ]:
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value

supply = {"P1": 800, "P2": 600}; demand = {"M1": 500, "M2": 400, "M3": 450}
km = {("P1","M1"):300, ("P1","M2"):700, ("P1","M3"):1100, ("P2","M1"):900, ("P2","M2"):400, ("P2","M3"):500}
rail_lanes = [k for k in km if km[k] >= 500]

def solve(cap=None, carbon_price=0):
    m = LpProblem("carbon", LpMinimize)
    truck = {k: m.add_variable(f"truck_{k[0]}_{k[1]}", lowBound=0) for k in km}
    rail = {k: m.add_variable(f"rail_{k[0]}_{k[1]}", lowBound=0, upBound=350) for k in rail_lanes}
    cost = lpSum(0.10 * km[k] * truck[k] for k in km) + lpSum((0.06 * km[k] + 30) * rail[k] for k in rail_lanes)
    co2 = (lpSum(0.080 * km[k] * truck[k] for k in km)
           + lpSum((0.025 * km[k] + 4.0) * rail[k] for k in rail_lanes)) * 0.001    # tons
    m += cost + carbon_price * co2
    for i in supply:
        m += lpSum(truck[k] for k in km if k[0] == i) + lpSum(rail[k] for k in rail_lanes if k[0] == i) <= supply[i]
    for j in demand:
        m += lpSum(truck[k] for k in km if k[1] == j) + lpSum(rail[k] for k in rail_lanes if k[1] == j) >= demand[j]
    if cap is not None:
        m += co2 <= cap, "co2_cap"
    m.solve(HiGHS(msg=False))
    price = [abs(c.pi) for c in m.constraints() if c.name == "co2_cap"]
    return cost.value(), co2.value(), (price[0] if price else 0)

print("Plans chosen at different carbon prices (the rows of the book's table):")
for carbon_price in (0, 60, 450):
    c, e, _ = solve(carbon_price=carbon_price)
    print(f"  price ${carbon_price}/t: cost ${c:,.0f}, CO2 {e:.1f} t/week")
print("Emissions-cap sweep (the abatement curve):")
for cap in [48.8, 45, 42, 40, 36, 34.2, 32]:
    c, e, p = solve(cap)
    print(f"  cap {cap}: cost ${c:,.0f}, cost of last ton ${p:,.0f}")


## Chapter 18: Reading model data from Excel and writing the plan back
Book answer: $5,650. Creates network.xlsx with sample data if it does not exist.

In [ ]:
import os
import pandas as pd

if not os.path.exists("network.xlsx"):
    with pd.ExcelWriter("network.xlsx") as w:
        pd.DataFrame({"plant": ["P1", "P2"], "supply": [500, 400]}).to_excel(w, sheet_name="supply", index=False)
        pd.DataFrame({"depot": ["D1", "D2", "D3"], "demand": [300, 350, 250]}).to_excel(w, sheet_name="demand", index=False)
        pd.DataFrame({"plant": ["P1"]*3 + ["P2"]*3, "depot": ["D1","D2","D3"]*2,
                      "cost": [8, 6, 10, 9, 7, 4]}).to_excel(w, sheet_name="costs", index=False)

from pulp import (LpProblem, LpMinimize, LpSolveStatus,
                  HiGHS, lpSum, value)

# Read the three sheets of the workbook
book = pd.read_excel("network.xlsx", sheet_name=None)
sup, dem = book["supply"], book["demand"]
lanes = book["costs"]
supply = dict(zip(sup["plant"], sup["supply"]))
demand = dict(zip(dem["depot"], dem["demand"]))
cost = {(r.plant, r.depot): r.cost
        for r in lanes.itertuples()}

# Basic data checks before building the model
assert not lanes.duplicated(["plant", "depot"]).any()
assert lanes.notna().all().all()
assert (lanes["cost"] >= 0).all()

model = LpProblem("distribution", LpMinimize)
x = {k: model.add_variable(f"x_{k[0]}_{k[1]}", lowBound=0)
     for k in cost}
model += lpSum(cost[k] * x[k] for k in cost)
for i in supply:
    out = [x[k] for k in cost if k[0] == i]
    model += lpSum(out) <= supply[i]
for j in demand:
    inbound = [x[k] for k in cost if k[1] == j]
    model += lpSum(inbound) >= demand[j]

result = model.solve(HiGHS(msg=False))
if result.status != LpSolveStatus.Optimal:
    raise SystemExit(f"Not optimal: {result.status.name}")

# Write the plan to a new workbook for the planners
lanes["ship"] = [value(x[(r.plant, r.depot)])
                 for r in lanes.itertuples()]
lanes.to_excel("plan.xlsx", index=False)
print("Total cost:", value(model.objective))


## Chapter 20: Meridian Foods fixed-charge location model
Book answers: open site B, total $1,546.5k per year. Peak plan 500/550/550/300 with no overtime, $38,750k.

In [ ]:
from pulp import LpProblem, LpMinimize, HiGHS, lpSum, value

regions = ["R1", "R2", "R3", "R4"]
demand = {"R1": 120, "R2": 95, "R3": 140, "R4": 85}          # thousands of units/year
dcs = ["DC0", "A", "B", "C"]
fixed = {"DC0": 0, "A": 300, "B": 260, "C": 240}               # $000/year
freight = {"DC0": {"R1":3.2, "R2":4.1, "R3":5.6, "R4":4.8},
           "A":   {"R1":2.9, "R2":3.0, "R3":3.4, "R4":5.5},
           "B":   {"R1":4.0, "R2":2.6, "R3":2.8, "R4":3.1},
           "C":   {"R1":3.1, "R2":3.8, "R3":4.4, "R4":3.0}}     # $/unit

m = LpProblem("meridian", LpMinimize)
y = {d: m.add_variable(f"open_{d}", cat="Binary") for d in dcs}
x = {(d, r): m.add_variable(f"ship_{d}_{r}", lowBound=0) for d in dcs for r in regions}
m += lpSum(fixed[d] * y[d] for d in dcs) + lpSum(freight[d][r] * x[d, r] for d in dcs for r in regions)
m += y["DC0"] == 1                                              # the existing DC stays open
for r in regions:
    m += lpSum(x[d, r] for d in dcs) == demand[r]
    for d in dcs:
        m += x[d, r] - demand[r] * y[d] <= 0
m.solve(HiGHS(msg=False))
print("Open:", [d for d in dcs if y[d].value() > 0.5])
for r in regions:
    print(f"  {r} served by", [d for d in dcs if x[d, r].value() > 1e-6])
print(f"Total annual cost: ${value(m.objective):,.1f}k")

# Step 4: planning for the peak (thousands of units)
demand, cap = [400, 500, 700, 300], 550
p = LpProblem("meridian_plan", LpMinimize)
T = range(4)
R = {t: p.add_variable(f"reg_{t}", lowBound=0, upBound=cap) for t in T}
O = {t: p.add_variable(f"ot_{t}", lowBound=0, upBound=100) for t in T}
I = {t: p.add_variable(f"inv_{t}", lowBound=0) for t in T}
p += lpSum(20 * R[t] + 32 * O[t] + 3 * I[t] for t in T)
for t in T:
    p += I[t] == (I[t - 1] if t else 0) + R[t] + O[t] - demand[t]
p.solve(HiGHS(msg=False))
print("Peak plan: regular", [R[t].value() for t in T], "overtime", [O[t].value() for t in T],
      f"cost ${value(p.objective):,.0f}k")


## Chapter 21: Last-mile delivery: daily routes versus alternate-day zones (Case A)
Exact routing by dynamic programming. Book answers: small vans daily $2,376/week, zones $2,671, large vans zones $1,567.

In [ ]:
import math
from functools import lru_cache

pts = {"DC": (0, 0), "C1": (4, 9), "C2": (-3, 8), "C3": (7, 4), "C4": (-8, 3), "C5": (2, -6),
       "C6": (-5, -7), "C7": (9, -3), "C8": (-1, 11), "C9": (6, -9), "C10": (-9, -4)}
daily = {"C1":6, "C2":5, "C3":7, "C4":4, "C5":6, "C6":5, "C7":8, "C8":4, "C9":5, "C10":6}
cust = list(daily); n = len(cust)
dist = lambda a, b: round(math.hypot(pts[a][0] - pts[b][0], pts[a][1] - pts[b][1]), 1)

# Shortest closed tour from the DC through every subset of customers (Held-Karp)
best = [[math.inf] * n for _ in range(1 << n)]
for i, c in enumerate(cust): best[1 << i][i] = dist("DC", c)
for mask in range(1 << n):
    for i in range(n):
        if best[mask][i] == math.inf: continue
        for j in range(n):
            if not mask >> j & 1:
                v = best[mask][i] + dist(cust[i], cust[j])
                if v < best[mask | 1 << j][j]: best[mask | 1 << j][j] = v
tour = [0] + [min(best[m][i] + dist(cust[i], "DC") for i in range(n) if m >> i & 1) for m in range(1, 1 << n)]

def plan(customers, capacity, mult, van_day, per_km):
    target = sum(1 << cust.index(c) for c in customers)
    load = lambda m: sum(daily[cust[i]] * mult for i in range(n) if m >> i & 1)
    @lru_cache(None)
    def f(m):
        if m == 0: return 0.0
        low, sub, out = m & -m, m, math.inf
        while sub:
            if sub & low and load(sub) <= capacity:
                out = min(out, van_day + per_km * tour[sub] + f(m ^ sub))
            sub = (sub - 1) & m
        return out
    return f(target)

north = [c for c in cust if pts[c][1] >= 0]; south = [c for c in cust if pts[c][1] < 0]
for name, cap, fix, km in [("30-crate vans", 30, 150, 1.2), ("60-crate vans", 60, 200, 1.5)]:
    daily_week = 6 * plan(cust, cap, 1, fix, km)
    zones_week = 3 * plan(north, cap, 2, fix, km) + 3 * plan(south, cap, 2, fix, km)
    print(f"{name}: daily ${daily_week:,.2f}/week, alternate-day zones ${zones_week:,.2f}/week")


## Chapter 21: Allocating a scarce shipment by marginal value (Case B)
Book answers: optimal 352/281/51/116 units, expected value $67,308 vs $64,490 for proportional allocation (284/231/196/89).

In [ ]:
from scipy.stats import norm

stores = {"Flagship": (320, 60, 99), "Mall": (260, 50, 89),
          "Outlet": (220, 110, 45), "Airport": (100, 25, 109)}   # mean, std dev, price
salvage, supply = 20, 800

def expected_sales(q, mu, sd):
    z = (q - mu) / sd
    return mu - sd * (norm.pdf(z) - z * (1 - norm.cdf(z)))
def value_of(alloc):
    return sum(p * expected_sales(q, mu, sd) + salvage * (q - expected_sales(q, mu, sd))
               for (mu, sd, p), q in ((stores[k], alloc[k]) for k in stores))
def marginal(k, q):
    mu, sd, p = stores[k]
    return salvage + (p - salvage) * (1 - norm.cdf((q + 0.5 - mu) / sd))

alloc = {k: 0 for k in stores}
for _ in range(supply):                       # give each unit to the store where it is worth most
    best = max(stores, key=lambda k: marginal(k, alloc[k])); alloc[best] += 1
proportional = {"Flagship": 284, "Mall": 231, "Outlet": 196, "Airport": 89}   # whole jackets
print("Optimal:", alloc, f"value ${value_of(alloc):,.0f}")
print("Proportional:", {k: round(v) for k, v in proportional.items()}, f"value ${value_of(proportional):,.0f}")
